In [1]:
import optuna
import numpy as np
import pandas as pd
import warnings
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

# 📌 Load dataset
df = pd.read_csv('/kaggle/input/binary-classification-with-a-bank-churn-dataset-1/train.csv')

# 📌 Drop Unnecessary Columns
df.drop(columns=["CustomerId", "Surname","Geography",], inplace=True)

# 📌 Encode Categorical Variables
label_encoder = LabelEncoder()
#df["Gender"] = label_encoder.fit_transform(df["Gender"])

# One-Hot Encoding for Geography
#df = pd.get_dummies(df, columns=["Geography"], drop_first=True)

# 📌 Feature Selection (Keep Only Important Features)
selected_features = [
    "CreditScore", "Gender", "Age", "Tenure", "Balance",
    "NumOfProducts", "HasCrCard", "IsActiveMember", "EstimatedSalary", "Geography_Germany", "Geography_Spain"
]







In [2]:
df

,id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,0,15665943.0,Nnamutaezinwa,616.0,Germany,Female,29.0,1.0,164947.05,2.0,0.0,1.0,183584.14,0.0
1,1,15623220.0,Yermakov,642.0,France,Female,29.0,7.0,0.00,2.0,1.0,1.0,139919.38,0.0
2,2,15690670.0,Hsieh,537.0,France,Male,38.0,1.0,86055.17,1.0,1.0,1.0,125422.66,0.0
3,3,15683053.0,Chidumaga,609.0,Germany,Female,34.0,2.0,105420.08,2.0,1.0,1.0,91366.42,0.0
4,4,15736228.0,Hsing,588.0,France,Female,35.0,4.0,0.00,2.0,1.0,1.0,151887.16,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14995,14995,15678944.0,Chidiebere,683.0,Germany,Female,38.0,6.0,135096.05,1.0,1.0,0.0,164083.72,1.0
14996,14996,15662294.0,Maughan,717.0,France,Female,39.0,2.0,0.00,2.0,1.0,1.0,94888.60,0.0
14997,14997,15602844.0,Shih,762.0,France,Female,39.0,2.0,0.00,2.0,1.0,1.0,131073.90,0.0
14998,14998,15612776.0,Hao,479.0,France,Female,36.0,3.0,129974.79,1.0,0.0,1.0,136497.28,0.0


In [81]:
import numpy as np
import pandas as pd
import warnings
import optuna
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")

# Load dataset
df = pd.read_csv('/kaggle/input/binary-classification-with-a-bank-churn-dataset-1/train.csv')

# Encoding categorical features
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])
df = df.drop(columns=['Surname'])
df = pd.get_dummies(df, columns=['Geography'], drop_first=False)

# Feature Engineering
df['Balance/Age'] = df['Balance'] / (df['Age'] + 1)
df['CreditScore/Age'] = df['CreditScore'] / (df['Age'] + 1)

# Outlier Removal with Isolation Forest
iso = IsolationForest(contamination=0.02, random_state=42)
outliers = iso.fit_predict(df.drop(columns=['Exited']))
df = df[outliers == 1]  # Remove anomalies

# Split data
X = df.drop(columns=['Exited'])
y = df['Exited']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ========================== OPTUNA HYPERPARAMETER TUNING ==========================

def tune_xgboost(trial):
    """Optimize XGBoost with Optuna"""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=100),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.1),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'eval_metric': 'logloss',
        'use_label_encoder': False
    }
    model = XGBClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in cv.split(X_train_scaled, y_train):
        model.fit(X_train_scaled[train_idx], y_train.iloc[train_idx])
        y_pred = model.predict_proba(X_train_scaled[val_idx])[:, 1]
        scores.append(roc_auc_score(y_train.iloc[val_idx], y_pred))
    
    return np.mean(scores)

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(tune_xgboost, n_trials=20)
best_xgb = XGBClassifier(**study_xgb.best_params)

# ---------------------- CatBoost Optimization ----------------------

def tune_catboost(trial):
    """Optimize CatBoost with Optuna"""
    params = {
        'iterations': trial.suggest_int('iterations', 500, 1000, step=100),
        'depth': trial.suggest_int('depth', 4, 8),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.03, 0.1),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 10.0),
        'verbose': 0
    }
    model = CatBoostClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in cv.split(X_train_scaled, y_train):
        model.fit(X_train_scaled[train_idx], y_train.iloc[train_idx])
        y_pred = model.predict_proba(X_train_scaled[val_idx])[:, 1]
        scores.append(roc_auc_score(y_train.iloc[val_idx], y_pred))
    
    return np.mean(scores)

study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(tune_catboost, n_trials=20)
best_cat = CatBoostClassifier(**study_cat.best_params)

# ---------------------- RandomForest Optimization ----------------------

def tune_rf(trial):
    """Optimize Random Forest with Optuna"""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=100),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5)
    }
    model = RandomForestClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in cv.split(X_train_scaled, y_train):
        model.fit(X_train_scaled[train_idx], y_train.iloc[train_idx])
        y_pred = model.predict_proba(X_train_scaled[val_idx])[:, 1]
        scores.append(roc_auc_score(y_train.iloc[val_idx], y_pred))
    
    return np.mean(scores)

study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(tune_rf, n_trials=20)
best_rf = RandomForestClassifier(**study_rf.best_params)

# ========================== STACKING CLASSIFIER ==========================

# Select the best model from Optuna tuning
best_models = {
    'XGB': (study_xgb.best_value, best_xgb),
    'CatBoost': (study_cat.best_value, best_cat),
    'RandomForest': (study_rf.best_value, best_rf)
}

best_model_name, (best_auc, best_model) = max(best_models.items(), key=lambda x: x[1][0])

print(f"\n🚀 Best Model for Final Estimator: {best_model_name} (ROC AUC: {best_auc:.4f})")

stacking_clf = StackingClassifier(
    estimators=[
        ('XGB', best_xgb),
        ('CatBoost', best_cat),
        ('RF', best_rf)
    ],
    final_estimator=best_model
)

# Train Stacking Model
stacking_clf.fit(X_train_scaled, y_train)

# ========================== TEST SET PREPROCESSING ==========================

test_df = pd.read_csv('/kaggle/input/binary-classification-with-a-bank-churn-dataset-1/test.csv')
test_df['Gender'] = le.transform(test_df['Gender'])
test_df = test_df.drop(columns=['Surname'])
test_df = pd.get_dummies(test_df, columns=['Geography'], drop_first=False)

# Ensure test set has same features as train set
df['Balance/Age'] = df['Balance'] / (df['Age'] + 1)
df['CreditScore/Age'] = df['CreditScore'] / (df['Age'] + 1)
for col in set(X_train.columns) - set(test_df.columns):
    test_df[col] = 0
test_df = test_df[X_train.columns]  # Reorder columns

X_test_final = scaler.transform(test_df)

# Predictions
y_test_probs = stacking_clf.predict_proba(X_test_final)[:, 1]

# Create Submission File
submission = pd.DataFrame({"id": test_df["id"], "Exited": y_test_probs})
submission.to_csv("submissions.csv", index=False)
print("\n✅ Submission file 'submissions.csv' saved successfully!")


[I 2025-02-17 20:13:14,896] A new study created in memory with name: no-name-fb9c1840-c044-45f7-88a6-a7b303135219
[I 2025-02-17 20:13:18,252] Trial 0 finished with value: 0.9264724710945897 and parameters: {'n_estimators': 400, 'learning_rate': 0.052392405566933496, 'max_depth': 6, 'subsample': 0.9762271750235858, 'colsample_bytree': 0.8041664689798609}. Best is trial 0 with value: 0.9264724710945897.
[I 2025-02-17 20:13:22,381] Trial 1 finished with value: 0.9238501327976568 and parameters: {'n_estimators': 500, 'learning_rate': 0.09286915510666956, 'max_depth': 4, 'subsample': 0.7010824420240647, 'colsample_bytree': 0.852885018693663}. Best is trial 0 with value: 0.9264724710945897.
[I 2025-02-17 20:13:23,604] Trial 2 finished with value: 0.9330897430952717 and parameters: {'n_estimators': 200, 'learning_rate': 0.02543978510018148, 'max_depth': 4, 'subsample': 0.635168481398026, 'colsample_bytree': 0.9977816308605394}. Best is trial 2 with value: 0.9330897430952717.
[I 2025-02-17 20:


🚀 Best Model for Final Estimator: XGB (ROC AUC: 0.9336)
0:	learn: 0.6584714	total: 3.86ms	remaining: 1.93s
1:	learn: 0.6294333	total: 7.6ms	remaining: 1.89s
2:	learn: 0.5990214	total: 10.9ms	remaining: 1.81s
3:	learn: 0.5695493	total: 14.7ms	remaining: 1.82s
4:	learn: 0.5446733	total: 18.4ms	remaining: 1.82s
5:	learn: 0.5224715	total: 22.2ms	remaining: 1.83s
6:	learn: 0.5014694	total: 26.7ms	remaining: 1.88s
7:	learn: 0.4835189	total: 30.6ms	remaining: 1.88s
8:	learn: 0.4693893	total: 33.8ms	remaining: 1.84s
9:	learn: 0.4537587	total: 37.6ms	remaining: 1.84s
10:	learn: 0.4408597	total: 41.1ms	remaining: 1.83s
11:	learn: 0.4276566	total: 44.6ms	remaining: 1.81s
12:	learn: 0.4158447	total: 48.3ms	remaining: 1.81s
13:	learn: 0.4052461	total: 51.9ms	remaining: 1.8s
14:	learn: 0.3961274	total: 56ms	remaining: 1.81s
15:	learn: 0.3897476	total: 60ms	remaining: 1.81s
16:	learn: 0.3807998	total: 63.7ms	remaining: 1.81s
17:	learn: 0.3735421	total: 67.5ms	remaining: 1.81s
18:	learn: 0.3663730	to

In [42]:
import optuna
import numpy as np
import pandas as pd
import warnings
import optuna.visualization as vis
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

# 📌 Load dataset
df = pd.read_csv('/kaggle/input/binary-classification-with-a-bank-churn-dataset-1/train.csv')

# 📌 Drop Unnecessary Columns
df.drop(columns=["CustomerId", "Surname"], inplace=True)

# 📌 Encode Categorical Variables
label_encoder = LabelEncoder()
df["Gender"] = label_encoder.fit_transform(df["Gender"])
df = pd.get_dummies(df, columns=['Geography'], drop_first=False)

# One-Hot Encoding for Geography

# 📌 Feature Selection
selected_features = [
    "CreditScore", "Gender", "Age", "Tenure", "Balance",
    "NumOfProducts", "HasCrCard", "IsActiveMember", "EstimatedSalary",
    "Geography_Spain","Geography_Germany","Geography_France"
]
# Outlier Removal with Isolation Forest
iso = IsolationForest(contamination=0.02, random_state=42)
outliers = iso.fit_predict(df.drop(columns=['Exited']))
df = df[outliers == 1]  # Remove anomalies
X = df[selected_features]
y = df["Exited"]

# 📌 Split Data (Fixing Previous Issue)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# 🔥 **OPTUNA HYPERPARAMETER TUNING**
def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 1000, 3000, step=100),
        "depth": trial.suggest_int("depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.1, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-1, 10, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-2, 1, log=True),
        "grow_policy": trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise"]),
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli"]),
        "early_stopping_rounds": 200,
        "verbose": 0,
        "loss_function": "Logloss"
    }

    if params["bootstrap_type"] == "Bayesian":
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0, 1)
    elif params["bootstrap_type"] == "Bernoulli":
        params["subsample"] = trial.suggest_float("subsample", 0.5, 1.0)

    # Train CatBoost with validation set
    model = CatBoostClassifier(**params, cat_features=["Gender", "Geography_Spain","Geography_Germany","Geography_France"])
    model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=200, verbose=0)

    # Validate model on validation set
    y_val_probs = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, y_val_probs)

# 🔍 Run Optuna Optimization
study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=100, n_jobs=-1)  # Increase trials for better tuning

# 📌 Retrieve Best Parameters
best_params = study.best_params
print("\n🔥 Best Hyperparameters for CatBoost:")
for key, value in best_params.items():
    print(f"{key}: {value}")

# 📊 Best Trial Details
best_trial = study.best_trial
print(f"\n🎯 Best AUC Score on Validation: {best_trial.value:.4f}")

# 📊 Visual Analysis
#vis.plot_optimization_history(study).show()
#vis.plot_param_importances(study).show()

best_catboost = CatBoostClassifier(**best_params, cat_features=["Gender", "Geography_Spain","Geography_Germany","Geography_France"])
best_catboost.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=100)

y_probs = best_catboost.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_probs)
print(f"\n🎯 Final ROC AUC on Test Set: {roc_auc:.4f}")

test_df = pd.read_csv('/kaggle/input/binary-classification-with-a-bank-churn-dataset-1/test.csv')

test_ids = test_df["id"]

test_df.drop(columns=["CustomerId", "Surname"], inplace=True)
test_df["Gender"] = label_encoder.transform(test_df["Gender"])
test_df = pd.get_dummies(test_df, columns=['Geography'], drop_first=False)

test_df = test_df[selected_features]
for col in set(X_train.columns) - set(test_df.columns):
    test_df[col] = 0  # Fill missing columns with 0
test_df = test_df[X_train.columns]  # Reorder columns

y_test_probs = best_catboost.predict_proba(test_df)[:, 1]

submission = pd.DataFrame({"id": test_ids, "Exited": y_test_probs})
submission.to_csv("submissions.csv", index=False)

print("\n✅ Submission file 'submissions.csv' saved successfully!")


[I 2025-02-20 14:06:49,013] A new study created in memory with name: no-name-940cfd72-4256-4019-99d3-3885e22a2469
[I 2025-02-20 14:06:53,787] Trial 0 finished with value: 0.9354612909302281 and parameters: {'iterations': 1200, 'depth': 8, 'learning_rate': 0.03287199939052434, 'l2_leaf_reg': 0.5638129456174177, 'random_strength': 0.4866375951622638, 'grow_policy': 'SymmetricTree', 'bootstrap_type': 'Bernoulli', 'subsample': 0.5355077325404094}. Best is trial 0 with value: 0.9354612909302281.
[I 2025-02-20 14:06:56,242] Trial 1 finished with value: 0.9369348181553715 and parameters: {'iterations': 2500, 'depth': 5, 'learning_rate': 0.011291056097860595, 'l2_leaf_reg': 0.16687453982363665, 'random_strength': 0.29838213348778486, 'grow_policy': 'SymmetricTree', 'bootstrap_type': 'Bernoulli', 'subsample': 0.6641216328423126}. Best is trial 1 with value: 0.9369348181553715.
[I 2025-02-20 14:06:59,537] Trial 2 finished with value: 0.9352227623173834 and parameters: {'iterations': 1400, 'depth


🔥 Best Hyperparameters for CatBoost:
iterations: 2800
depth: 5
learning_rate: 0.0020135802980959716
l2_leaf_reg: 2.2606430920709037
random_strength: 0.062000031899527866
grow_policy: SymmetricTree
bootstrap_type: Bernoulli
subsample: 0.9798461168079672

🎯 Best AUC Score on Validation: 0.9372
0:	learn: 0.6906829	test: 0.6906692	best: 0.6906692 (0)	total: 6.99ms	remaining: 19.6s
100:	learn: 0.5075683	test: 0.5065373	best: 0.5065373 (100)	total: 393ms	remaining: 10.5s
200:	learn: 0.4092740	test: 0.4076188	best: 0.4076188 (200)	total: 740ms	remaining: 9.56s
300:	learn: 0.3537892	test: 0.3517469	best: 0.3517469 (300)	total: 1.08s	remaining: 8.98s
400:	learn: 0.3205863	test: 0.3184622	best: 0.3184622 (400)	total: 1.48s	remaining: 8.88s
500:	learn: 0.2996292	test: 0.2975312	best: 0.2975312 (500)	total: 1.86s	remaining: 8.54s
600:	learn: 0.2857558	test: 0.2834870	best: 0.2834870 (600)	total: 2.21s	remaining: 8.1s
700:	learn: 0.2761880	test: 0.2737055	best: 0.2737055 (700)	total: 2.56s	remaini

In [43]:
best_catboost = CatBoostClassifier(**best_params, )#cat_features=["Gender", "Geography_Spain","Geography_Germany","Geography_France"])
best_catboost.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=100)

y_probs = best_catboost.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_probs)
print(f"\n🎯 Final ROC AUC on Test Set: {roc_auc:.4f}")

test_df = pd.read_csv('/kaggle/input/binary-classification-with-a-bank-churn-dataset-1/test.csv')

test_ids = test_df["id"]

test_df.drop(columns=["CustomerId", "Surname"], inplace=True)
test_df["Gender"] = label_encoder.transform(test_df["Gender"])
test_df = pd.get_dummies(test_df, columns=['Geography'], drop_first=False)

test_df = test_df[selected_features]
for col in set(X_train.columns) - set(test_df.columns):
    test_df[col] = 0  # Fill missing columns with 0
test_df = test_df[X_train.columns]  # Reorder columns

y_test_probs = best_catboost.predict_proba(test_df)[:, 1]

submission = pd.DataFrame({"id": test_ids, "Exited": y_test_probs})
submission.to_csv("submissions.csv", index=False)

print("\n✅ Submission file 'submissions.csv' saved successfully!")

0:	learn: 0.6906860	test: 0.6906804	best: 0.6906804 (0)	total: 3.54ms	remaining: 9.9s
100:	learn: 0.5074186	test: 0.5064034	best: 0.5064034 (100)	total: 342ms	remaining: 9.15s
200:	learn: 0.4091648	test: 0.4074716	best: 0.4074716 (200)	total: 681ms	remaining: 8.81s
300:	learn: 0.3538155	test: 0.3516552	best: 0.3516552 (300)	total: 1.03s	remaining: 8.53s
400:	learn: 0.3206019	test: 0.3184778	best: 0.3184778 (400)	total: 1.37s	remaining: 8.22s
500:	learn: 0.2997075	test: 0.2975678	best: 0.2975678 (500)	total: 1.72s	remaining: 7.89s
600:	learn: 0.2856692	test: 0.2833734	best: 0.2833734 (600)	total: 2.06s	remaining: 7.53s
700:	learn: 0.2761749	test: 0.2736960	best: 0.2736960 (700)	total: 2.4s	remaining: 7.18s
800:	learn: 0.2695609	test: 0.2670080	best: 0.2670080 (800)	total: 2.73s	remaining: 6.82s
900:	learn: 0.2645583	test: 0.2620000	best: 0.2620000 (900)	total: 3.07s	remaining: 6.47s
1000:	learn: 0.2606984	test: 0.2582092	best: 0.2582092 (1000)	total: 3.41s	remaining: 6.13s
1100:	learn: 

In [46]:
import optuna
import numpy as np
import pandas as pd
import warnings
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# 📌 Load dataset
df = pd.read_csv('/kaggle/input/binary-classification-with-a-bank-churn-dataset-1/train.csv')

# 📌 Encode categorical variables
df['Gender'] = LabelEncoder().fit_transform(df['Gender'])
df = df.drop(columns=['Surname'])

# 📌 One-hot encode 'Geography' column
df = pd.get_dummies(df, columns=['Geography'])

# 📌 Feature Engineering
df['Balance/Age'] = df['Balance'] / (df['Age'] + 1)
df['CreditScore/Age'] = df['CreditScore'] / (df['Age'] + 1)

# 📌 Split Data
X = df.drop(columns=['Exited'])
y = df['Exited']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 📌 Standardize numerical features
scaler = StandardScaler()
num_features = X_train.select_dtypes(include=['float64', 'int64']).columns
X_train[num_features] = scaler.fit_transform(X_train[num_features])
X_test[num_features] = scaler.transform(X_test[num_features])

# 🔥 **OPTUNA HYPERPARAMETER TUNING FUNCTIONS**
def tune_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'eval_metric': 'logloss',
        'use_label_encoder': False,
        'enable_categorical': True  # ✅ Fix for categorical data
    }
    model = XGBClassifier(**params)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    return cross_val_score(model, X_train, y_train, cv=skf, scoring='roc_auc', n_jobs=-1).mean()

def tune_catboost(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 500, 2000, step=100),
        'depth': trial.suggest_int('depth', 4, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 10, log=True),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'random_strength': trial.suggest_float('random_strength', 1e-2, 10, log=True),
        'bootstrap_type': trial.suggest_categorical('bootstrap_type', ['Bayesian', 'Bernoulli', 'MVS']),
        'verbose': 0,
        'loss_function': 'Logloss'
    }
    model = CatBoostClassifier(**params)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    return cross_val_score(model, X_train, y_train, cv=skf, scoring='roc_auc', n_jobs=-1).mean()

def tune_random_forest(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5)
    }
    model = RandomForestClassifier(**params)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    return cross_val_score(model, X_train, y_train, cv=skf, scoring='roc_auc', n_jobs=-1).mean()

# 🔍 **Run Optuna Optimization**
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(tune_xgb, n_trials=1)

study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(tune_catboost, n_trials=50)

study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(tune_random_forest, n_trials=1)

# 📌 Best Hyperparameters
best_xgb_params = study_xgb.best_params
best_cat_params = study_cat.best_params
best_rf_params = study_rf.best_params

print("\n🔥 Best XGBoost Params:", best_xgb_params)
print("\n🔥 Best CatBoost Params:", best_cat_params)
print("\n🔥 Best Random Forest Params:", best_rf_params)

# 📌 Initialize Optimized Models
xgb_model = XGBClassifier(**best_xgb_params, eval_metric='logloss', use_label_encoder=False, enable_categorical=True)
cat_model = CatBoostClassifier(**best_cat_params)
rf_model = RandomForestClassifier(**best_rf_params)

# 📌 Train Stacking Classifier
stacking_clf = StackingClassifier(
    estimators=[
        ('XGB', xgb_model),
        ('CatBoost', cat_model),
        ('RF', rf_model)
    ],
    final_estimator=LogisticRegression()
)

stacking_clf.fit(X_train, y_train)

# 📊 Evaluate Stacking Model
y_probs = stacking_clf.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_probs)
print(f"\n🎯 Stacking Classifier ROC AUC: {roc_auc:.4f}")

# 📌 Process Test Dataset
test_df = pd.read_csv('/kaggle/input/binary-classification-with-a-bank-churn-dataset-1/test.csv')
test_df['Gender'] = LabelEncoder().fit_transform(test_df['Gender'])
test_df = test_df.drop(columns=['Surname'])

# One-hot encode 'Geography' in test set
test_df = pd.get_dummies(test_df, columns=['Geography'])

# Ensure test set has same features as train set
missing_cols = set(X_train.columns) - set(test_df.columns)
for col in missing_cols:
    test_df[col] = 0
test_df = test_df[X_train.columns]  # Reorder columns

# Standardize numerical features
test_df[num_features] = scaler.transform(test_df[num_features])

# 📌 Make Predictions
y_test_probs = stacking_clf.predict_proba(test_df)[:, 1]

# 📌 Create Submission File
submission = pd.DataFrame({
    "id": test_df["id"],  # Ensure correct ID column
    "Exited": y_test_probs
})
submission.to_csv("submissions.csv", index=False)
print("\n✅ Submission file 'submissions.csv' saved successfully!")


[I 2025-02-20 14:35:26,496] A new study created in memory with name: no-name-02b0e698-3088-4fdf-acd2-3f2327bd9b08
[I 2025-02-20 14:35:27,450] Trial 0 finished with value: 0.9333182426104717 and parameters: {'n_estimators': 250, 'learning_rate': 0.03524525912919723, 'max_depth': 3, 'subsample': 0.8893323925636398, 'colsample_bytree': 0.5162752549097177}. Best is trial 0 with value: 0.9333182426104717.
[I 2025-02-20 14:35:27,451] A new study created in memory with name: no-name-cfa72f09-8077-46e0-915b-165c5452412b
[I 2025-02-20 14:37:18,139] Trial 0 finished with value: 0.9283557283375288 and parameters: {'iterations': 700, 'depth': 12, 'learning_rate': 0.008985272982826783, 'l2_leaf_reg': 4.9659373099713715, 'border_count': 122, 'random_strength': 0.022800177392225174, 'bootstrap_type': 'Bernoulli'}. Best is trial 0 with value: 0.9283557283375288.
[I 2025-02-20 14:39:30,304] Trial 1 finished with value: 0.9210005388369173 and parameters: {'iterations': 1800, 'depth': 11, 'learning_rate'


🔥 Best XGBoost Params: {'n_estimators': 250, 'learning_rate': 0.03524525912919723, 'max_depth': 3, 'subsample': 0.8893323925636398, 'colsample_bytree': 0.5162752549097177}

🔥 Best CatBoost Params: {'iterations': 800, 'depth': 5, 'learning_rate': 0.008299669202057247, 'l2_leaf_reg': 2.219766601601176, 'border_count': 235, 'random_strength': 0.7460809859527414, 'bootstrap_type': 'MVS'}

🔥 Best Random Forest Params: {'n_estimators': 300, 'max_depth': 9, 'min_samples_split': 5, 'min_samples_leaf': 1}
0:	learn: 0.6835695	total: 8.7ms	remaining: 6.95s
1:	learn: 0.6743270	total: 13.2ms	remaining: 5.26s
2:	learn: 0.6652061	total: 17.8ms	remaining: 4.74s
3:	learn: 0.6558944	total: 22.5ms	remaining: 4.47s
4:	learn: 0.6471602	total: 26.7ms	remaining: 4.25s
5:	learn: 0.6383695	total: 31.2ms	remaining: 4.13s
6:	learn: 0.6301684	total: 35.6ms	remaining: 4.04s
7:	learn: 0.6219593	total: 40ms	remaining: 3.96s
8:	learn: 0.6142070	total: 44.7ms	remaining: 3.92s
9:	learn: 0.6061809	total: 49.3ms	remaini